# Display Benefits From Same Product

### Initiating DB connection, and create a connection function.

In [4]:
# Install required libraries (run this once)
!pip install paramiko pymysql sshtunnel
!pip install python-dotenv

# Import required libraries
import paramiko
import pymysql
from sshtunnel import SSHTunnelForwarder

# Import required library for reading .env file
from dotenv import load_dotenv
import os

# Configuration for SSH
SSH_HOST = os.getenv("SSH_HOST")  # Read from .env file
SSH_PORT = int(os.getenv("SSH_PORT", 38123))  # Default SSH port if not specified
SSH_USER = os.getenv("SSH_USER")  # Read from .env file
SSH_KEY = os.getenv("SSH_KEY")  # Read from .env file
SSH_PASSPHRASE = os.getenv("SSH_PASSPHRASE")  # Read from .env file

# Configuration for MySQL
MYSQL_HOST = os.getenv("MYSQL_HOST")  # Read from .env file
MYSQL_PORT = int(os.getenv("MYSQL_PORT", 3306))  # Default MySQL port if not specified
MYSQL_USER = os.getenv("MYSQL_USER")  # Read from .env file
MYSQL_PASSWORD = os.getenv("MYSQL_PASSWORD")  # Read from .env file
MYSQL_DB = 'qoala_finance_service_development'      # MySQL database name

def create_db_connection():
    """
    Creates and returns a MySQL database connection through an SSH tunnel.
    
    Returns:
        connection: A MySQL database connection object.
    """
    try:
        # Create an SSH tunnel
        tunnel = SSHTunnelForwarder(
            (SSH_HOST, SSH_PORT),
            ssh_username=SSH_USER,
            ssh_pkey=SSH_KEY,
            ssh_private_key_password=SSH_PASSPHRASE,
            remote_bind_address=(MYSQL_HOST, MYSQL_PORT)
        )
        tunnel.start()
        
        # Connect to MySQL through the SSH tunnel
        connection = pymysql.connect(
            host='127.0.0.1',  # Localhost because of the tunnel
            port=tunnel.local_bind_port,
            user=MYSQL_USER,
            password=MYSQL_PASSWORD,
            db=MYSQL_DB
        )
        
        return connection, tunnel
    
    except Exception as e:
        print(f"An error occurred while creating the connection: {e}")
        return None, None

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 24.2 -> 25.0.1
[notice] To update, run: /Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip
Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 24.2 -> 25.0.1
[notice] To update, run: /Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip


### Display benefits that have same product_id from claim.

In [5]:
def execute_sql_query(connection, query, params=None):
    """
    Executes an SQL query using an existing database connection.
    
    Args:
        connection: A MySQL database connection object.
        query (str): The SQL query to execute.
        params (tuple, optional): Parameters to safely inject into the query.
    
    Returns:
        list: A list of tuples (or dictionaries) containing the query results.
    """
    try:
        # Create a cursor object (use DictCursor for dictionary results)
        cursor = connection.cursor(pymysql.cursors.DictCursor)
        
        # Execute the SQL query with parameters
        if params:
            cursor.execute(query, params)
        else:
            cursor.execute(query)
        
        # Fetch the results
        results = cursor.fetchall()
        
        # Close the cursor (but leave the connection open)
        cursor.close()
        
        return results
    
    except Exception as e:
        print(f"An error occurred while executing the query: {e}")
        return None

# Create a connection
connection, tunnel = create_db_connection()

if connection:
    try:
        # Example query
        query = """
        SELECT b.product_id, b.code, b.name 
        FROM benefits b
        INNER JOIN benefits bs ON b.product_id = bs.product_id
        WHERE bs.code = %s
        ORDER BY b.code;
        """
        
        # Get user input and validate it
        user_input = input("Please enter the current benefit code: ").strip()
        # conditional = "FL-73"
        if not user_input:
            print("Error: Benefit code cannot be empty.")
        else:
            # Convert params to tuple correctly
            params = (user_input,)
            # params = (conditional,)
            # Execute the query
            results = execute_sql_query(connection, query, params)  # ✅ Fixed function call
            
            if results:
                print("Query Results:")
                for row in results:
                    print(row)
            else:
                print("No results returned.")

    except Exception as e:
        print(f"An unexpected error occurred: {e}")

    finally:
        # Close the connection and tunnel when done
        if connection:
            connection.close()
        if tunnel:
            tunnel.stop()
else:
    print("Failed to establish a database connection.")

Error: Benefit code cannot be empty.
